In [ ]:
! pip install scenedetect[opencv]


In [ ]:
import cv2
import json
import shutil
import os
from pathlib import Path
from google.colab import drive
from scenedetect import open_video, SceneManager
from scenedetect.detectors import AdaptiveDetector

# 1. Mount Drive
drive.mount('/content/drive')

# Video PATH trên Drive
VIDEO_PATH_TO_TEST = Path("/content/drive/MyDrive/DACNTT_voice/Data/aic")

# Output PATH trên Drive
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/DACNTT_voice/Data")

# Thư mục tạm trên Colab để xử lý cho nhanh
TEMP_ROOT = Path("/content/temp_processing")

Mounted at /content/drive


In [ ]:
import os
import sys

# 1. Định nghĩa lại các đường dẫn quan trọng
DRIVE_BASE = Path("/content/drive/MyDrive/DACNTT_voice")
DRIVE_DATA_ROOT = DRIVE_BASE / "Data"

# 2. HÀM KIỂM TRA SHORTCUT VÀ ĐƯỜNG DẪN
def check_drive_setup():
    print("--- Đang kiểm tra hệ thống đường dẫn ---")

    # Kiểm tra Shortcut gốc
    if not DRIVE_BASE.exists():
        print(f"LỖI: Không tìm thấy Shortcut tại: {DRIVE_BASE}")
        print(" Hướng dẫn: Hãy xóa Shortcut cũ, vào 'Shared with me', tạo lại Shortcut mới vào My Drive.")
        return False

    # Kiểm tra xem có phải là Shortcut bị lỗi (mờ) không bằng cách thử liệt kê file
    try:
        os.listdir(DRIVE_BASE)
    except Exception as e:
        print(f" LỖI: Shortcut tồn tại nhưng không thể truy cập (có thể bị mờ hoặc mất quyền): {e}")
        return False

    # Kiểm tra thư mục Data bên trong
    if not DRIVE_DATA_ROOT.exists():
        print(f" Không tìm thấy thư mục 'Data' trong {DRIVE_BASE.name}")

    print("✅ Hệ thống đường dẫn OK. Sẵn sàng chạy code!")
    return True

# 3. CHẠY KIỂM TRA TRƯỚC KHI VÀO LUỒNG CHÍNH
if not check_drive_setup():
    # Nếu kiểm tra thất bại, dừng toàn bộ code để tránh lỗi lưu file lung tung
    sys.exit("Dừng chương trình do lỗi đường dẫn Drive.")

--- Đang kiểm tra hệ thống đường dẫn ---
✅ Hệ thống đường dẫn OK. Sẵn sàng chạy code!


In [ ]:

def detect_scenes(video_path, min_scene_len_sec=1.0):
    """Phát hiện cảnh quay bằng PySceneDetect."""
    video = open_video(str(video_path))
    fps = video.frame_rate

    scene_manager = SceneManager()
    scene_manager.add_detector(AdaptiveDetector(
        adaptive_threshold=3.0,
        min_scene_len=int(fps * min_scene_len_sec),
    ))
    scene_manager.detect_scenes(video)
    scene_list = scene_manager.get_scene_list()

    if not scene_list:
        return [(0.0, round(video.duration.get_seconds(), 3))]

    return [(round(s.get_seconds(), 3), round(e.get_seconds(), 3)) for s, e in scene_list]

def extract_middle_frame(video_path, time_sec):
    """Trích xuất 1 khung hình tại thời điểm chỉ định."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_idx = int(time_sec * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None

def process_video(video_path: Path):
    parent_folder = video_path.parent.name
    clip_name = video_path.stem
    output_stem = f"{parent_folder}_{clip_name}"

    # Nếu file JSON đã có trên Drive thì bỏ qua không xử lý lại video này nữa
    final_json_path = DRIVE_DATA_ROOT / "time_extract" / parent_folder / f"{output_stem}.json"
    if final_json_path.exists():
        print(f" Bỏ qua (Đã tồn tại): {video_path.name}")
        return

    # Tạo đường dẫn tạm trên SSD Colab
    temp_frame_dir = TEMP_ROOT / "frame_extracted" / parent_folder / output_stem
    temp_json_dir = TEMP_ROOT / "time_extract" / parent_folder
    temp_frame_dir.mkdir(parents=True, exist_ok=True)
    temp_json_dir.mkdir(parents=True, exist_ok=True)

    scenes = detect_scenes(video_path)
    results = []

    for start, end in scenes:
        mid_time = (start + end) / 2
        frame = extract_middle_frame(video_path, mid_time)
        if frame is not None:
            img_name = f"{int(mid_time):03d}.jpg"
            img_path = temp_frame_dir / img_name
            cv2.imwrite(str(img_path), frame)

            # Lưu đường dẫn tương đối vào JSON
            results.append({
                "start": start,
                "end": end,
                "image": f"frame_extracted/{parent_folder}/{output_stem}/{img_name}"
            })

    with open(temp_json_dir / f"{output_stem}.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

def sync_to_drive(parent_folder_name):
    """Đồng bộ từ SSD Colab sang Google Drive"""
    print(f">>> Đang đồng bộ {parent_folder_name} sang Drive...")

    # Đồng bộ Ảnh
    src_f = TEMP_ROOT / "frame_extracted" / parent_folder_name
    dst_f = DRIVE_DATA_ROOT / "frame_extracted" / parent_folder_name
    if src_f.exists():
        if dst_f.exists(): shutil.rmtree(dst_f)
        shutil.copytree(src_f, dst_f)

    # Đồng bộ JSON
    src_j = TEMP_ROOT / "time_extract" / parent_folder_name
    dst_j = DRIVE_DATA_ROOT / "time_extract" / parent_folder_name
    if src_j.exists():
        if dst_j.exists(): shutil.rmtree(dst_j)
        shutil.copytree(src_j, dst_j)

    print(f"Xong thư mục: {parent_folder_name}")

def run_main():
    if not VIDEO_PATH_TO_TEST.exists():
        print(f"Lỗi: Không tìm thấy thư mục {VIDEO_PATH_TO_TEST}")
        return

    # Lấy TẤT CẢ các thư mục con
    folders = sorted([f for f in VIDEO_PATH_TO_TEST.iterdir() if f.is_dir()])

    if not folders:
        print(f"Thư mục {VIDEO_PATH_TO_TEST.name} trống, không có thư mục con nào.")
        return

    print(f"Tìm thấy {len(folders)} thư mục. Bắt đầu xử lý...\n")

    for folder in folders:
        videos = sorted([v for v in folder.glob("*.mp4")])
        if not videos: continue

        print(f" Đang xử lý thư mục: {folder.name} ({len(videos)} videos)")

        for v in videos:
            try:
                process_video(v)
                print(f"  Đã xong: {v.name}")
            except Exception as e:
                print(f"   Lỗi tại file {v.name}: {e}")

        # Đồng bộ sau khi xong mỗi folder
        sync_to_drive(folder.name)

        # Giải phóng SSD Colab ngay sau khi đồng bộ để tránh đầy bộ nhớ
        shutil.rmtree(TEMP_ROOT / "frame_extracted" / folder.name, ignore_errors=True)
        shutil.rmtree(TEMP_ROOT / "time_extract" / folder.name, ignore_errors=True)


if __name__ == "__main__":
    run_main()

Streaming output truncated to the last 5000 lines.
 Bỏ qua (Đã tồn tại): 004.mp4
  Đã xong: 004.mp4
 Bỏ qua (Đã tồn tại): 005.mp4
  Đã xong: 005.mp4
 Bỏ qua (Đã tồn tại): 006.mp4
  Đã xong: 006.mp4
 Bỏ qua (Đã tồn tại): 007.mp4
  Đã xong: 007.mp4
 Bỏ qua (Đã tồn tại): 008.mp4
  Đã xong: 008.mp4
 Bỏ qua (Đã tồn tại): 009.mp4
  Đã xong: 009.mp4
 Bỏ qua (Đã tồn tại): 010.mp4
  Đã xong: 010.mp4
 Bỏ qua (Đã tồn tại): 011.mp4
  Đã xong: 011.mp4
 Bỏ qua (Đã tồn tại): 012.mp4
  Đã xong: 012.mp4
 Bỏ qua (Đã tồn tại): 013.mp4
  Đã xong: 013.mp4
 Bỏ qua (Đã tồn tại): 014.mp4
  Đã xong: 014.mp4
 Bỏ qua (Đã tồn tại): 015.mp4
  Đã xong: 015.mp4
 Bỏ qua (Đã tồn tại): 016.mp4
  Đã xong: 016.mp4
 Bỏ qua (Đã tồn tại): 017.mp4
  Đã xong: 017.mp4
 Bỏ qua (Đã tồn tại): 018.mp4
  Đã xong: 018.mp4
>>> Đang đồng bộ K18_V022 sang Drive...
Xong thư mục: K18_V022
 Đang xử lý thư mục: K18_V023 (18 videos)
 Bỏ qua (Đã tồn tại): 000.mp4
  Đã xong: 000.mp4
 Bỏ qua (Đã tồn tại): 001.mp4
  Đã xong: 001.mp4
 Bỏ qua (Đã